# Workshop 3 — Data Quality Analysis & Gold Layer Exploration
**NYC Gastronomic Trends Sentiment Analysis**  
Data Analysis Programming — Semester 2026-I  

This notebook covers:
1. Silver layer data quality findings (null rates, duplicates, text statistics)
2. Gold layer governance KPI table and visualizations
3. Gold layer storytelling summary exploration
4. Structured data quality findings report

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import glob
import warnings
warnings.filterwarnings('ignore')

# ── Light style ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#ffffff',
    'axes.facecolor':   '#f8f8f8',
    'axes.edgecolor':   '#cccccc',
    'axes.labelcolor':  '#222222',
    'xtick.color':      '#555555',
    'ytick.color':      '#555555',
    'text.color':       '#222222',
    'grid.color':       '#dddddd',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'monospace',
    'font.size':        10,
})
NEON   = '#e6a800'   # amber — visible on white
HEAT   = '#cc3333'   # red
GREEN  = '#1a7a40'   # dark green
SMOKE  = '#888888'   # mid-grey
CHALK  = '#111111'   # near-black (text on white)
BG     = '#ffffff'

print('Libraries loaded — light theme active.')

---
## 1. Load Silver Layer Data

In [ ]:
BASE = '..'

# ── API (Spoonacular recipes) ────────────────────────────────────────────────
api_files = sorted(glob.glob(f'{BASE}/datalake_silver/api/**/*.parquet', recursive=True))
df_api = pd.concat([pd.read_parquet(f) for f in api_files], ignore_index=True)

# ── Web scraping (Eater NY articles) ────────────────────────────────────────
web_files = sorted(glob.glob(f'{BASE}/datalake_silver/webscraping/**/*.parquet', recursive=True))
df_web = pd.concat([pd.read_parquet(f) for f in web_files], ignore_index=True)

print(f'Silver API   → {len(api_files)} files | {len(df_api):,} records | {len(df_api.columns)} columns')
print(f'Silver Web   → {len(web_files)} files | {len(df_web):,} records | {len(df_web.columns)} columns')

In [ ]:
print('=== Silver API — schema ===')
df_api.dtypes

In [ ]:
print('=== Silver Web — schema ===')
df_web.dtypes

---
## 2. Silver Data Quality — Null Rate Analysis

In [ ]:
def null_rate(df):
    """Return null rate % per column, sorted descending."""
    return (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

nr_api = null_rate(df_api)
nr_web = null_rate(df_web)

# ── Summary table ────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Source':        ['API (Spoonacular)', 'Web (Eater NY)'],
    'Records':       [len(df_api), len(df_web)],
    'Columns':       [len(df_api.columns), len(df_web.columns)],
    'Fields w/ nulls': [(nr_api > 0).sum(), (nr_web > 0).sum()],
    'Max null rate': [f"{nr_api.max():.1f}%", f"{nr_web.max():.1f}%"],
    'Worst field':   [nr_api.idxmax(), nr_web.idxmax()],
})
print('=== NULL RATE OVERVIEW ===')
print(summary.to_string(index=False))

In [ ]:
# ── Figure 1: Null rates per field — API source ──────────────────────────────
nr_api_nonzero = nr_api[nr_api > 0]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Figure 1 — Null Rate per Field (Silver Layer)', fontsize=13, color=CHALK, fontweight='bold', y=1.01)

# Left: API
ax = axes[0]
if len(nr_api_nonzero) > 0:
    colors_api = [HEAT if v > 20 else NEON if v > 5 else SMOKE for v in nr_api_nonzero.values]
    bars = ax.barh(nr_api_nonzero.index, nr_api_nonzero.values, color=colors_api, edgecolor='none', height=0.6)
    for bar, val in zip(bars, nr_api_nonzero.values):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=8, color=CHALK)
    ax.set_title('Source: API (Spoonacular)', color=CHALK, fontsize=10, fontweight='bold')
    ax.set_xlabel('Null Rate (%)', color=SMOKE)
    ax.axvline(x=5, color=NEON, linestyle='--', alpha=0.6, linewidth=0.9, label='5% threshold')
    ax.axvline(x=20, color=HEAT, linestyle='--', alpha=0.6, linewidth=0.9, label='20% threshold')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'No null fields found', ha='center', va='center', transform=ax.transAxes, color=GREEN, fontsize=12)
    ax.set_title('Source: API (Spoonacular)', color=CHALK, fontsize=10, fontweight='bold')
ax.grid(axis='x')

# Right: Web
ax2 = axes[1]
nr_web_nonzero = nr_web[nr_web > 0]
if len(nr_web_nonzero) > 0:
    colors_web = [HEAT if v > 20 else NEON if v > 5 else SMOKE for v in nr_web_nonzero.values]
    bars2 = ax2.barh(nr_web_nonzero.index, nr_web_nonzero.values, color=colors_web, edgecolor='none', height=0.6)
    for bar, val in zip(bars2, nr_web_nonzero.values):
        ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=8, color=CHALK)
    ax2.set_title('Source: Web Scraping (Eater NY)', color=CHALK, fontsize=10, fontweight='bold')
    ax2.set_xlabel('Null Rate (%)', color=SMOKE)
    ax2.axvline(x=5, color=NEON, linestyle='--', alpha=0.6, linewidth=0.9)
    ax2.axvline(x=20, color=HEAT, linestyle='--', alpha=0.6, linewidth=0.9)
else:
    ax2.text(0.5, 0.5, 'No null fields found', ha='center', va='center', transform=ax2.transAxes, color=GREEN, fontsize=12)
    ax2.set_title('Source: Web Scraping (Eater NY)', color=CHALK, fontsize=10, fontweight='bold')
ax2.grid(axis='x')

plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/fig1_null_rates.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: fig1_null_rates.png')

---
## 3. Silver Data Quality — Duplicate Detection

In [ ]:
# ── Duplicate analysis ───────────────────────────────────────────────────────
# Some parquet columns contain list/array values that pandas cannot hash.
# Filter to scalar-typed columns only before calling duplicated().
def hashable_cols(df):
    cols = []
    for col in df.columns:
        first = df[col].dropna().iloc[0] if df[col].dropna().shape[0] > 0 else None
        if first is None or not isinstance(first, (list, dict, np.ndarray)):
            cols.append(col)
    return cols

dup_api = df_api[hashable_cols(df_api)].duplicated().sum()
dup_web = df_web[hashable_cols(df_web)].duplicated().sum()

# ID-level duplicates for API
if 'id' in df_api.columns:
    dup_api_id = df_api['id'].duplicated().sum()
else:
    dup_api_id = 0

# URL-level duplicates for web
if 'article_url' in df_web.columns:
    dup_web_url = df_web['article_url'].duplicated().sum()
else:
    dup_web_url = 0

dup_summary = pd.DataFrame({
    'Source':             ['API', 'Web Scraping'],
    'Total records':      [len(df_api), len(df_web)],
    'Full duplicates':    [dup_api,    dup_web],
    'Full dup rate':      [f'{dup_api/len(df_api)*100:.2f}%',    f'{dup_web/len(df_web)*100:.2f}%'],
    'ID/URL duplicates':  [dup_api_id, dup_web_url],
    'ID/URL dup rate':    [f'{dup_api_id/len(df_api)*100:.2f}%', f'{dup_web_url/len(df_web)*100:.2f}%'],
})
print('=== DUPLICATE ANALYSIS ===')
print(dup_summary.to_string(index=False))

In [ ]:
# ── Figure 2: Duplicate rate comparison ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle('Figure 2 — Duplicate Rate by Source', fontsize=13, color=CHALK, fontweight='bold')

sources    = ['API (Spoonacular)', 'Web (Eater NY)']
full_rates = [dup_api/len(df_api)*100, dup_web/len(df_web)*100]
id_rates   = [dup_api_id/len(df_api)*100, dup_web_url/len(df_web)*100]

x = np.arange(len(sources))
w = 0.35
b1 = ax.bar(x - w/2, full_rates, w, label='Full row duplicates', color=HEAT,  alpha=0.80, edgecolor='none')
b2 = ax.bar(x + w/2, id_rates,   w, label='ID / URL duplicates', color=NEON, alpha=0.80, edgecolor='none')

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.2f}%', ha='center', va='bottom', fontsize=8, color=CHALK)

ax.set_xticks(x)
ax.set_xticklabels(sources)
ax.set_ylabel('Duplicate Rate (%)')
ax.set_ylim(0, max(max(full_rates), max(id_rates)) * 1.4 + 2)
ax.axhline(y=5, color=SMOKE, linestyle='--', alpha=0.5, linewidth=0.8, label='5% acceptable threshold')
ax.legend(fontsize=8)
ax.grid(axis='y')

plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/fig2_duplicates.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 4. Silver Data Quality — Text Length Statistics

In [ ]:
# Text length analysis
text_cols = {
    'API — summary':              df_api['summary'].dropna().str.len() if 'summary' in df_api.columns else pd.Series(dtype=float),
    'API — instructions_text':    df_api['instructions_text'].dropna().str.len() if 'instructions_text' in df_api.columns else pd.Series(dtype=float),
    'Web — article_summary':      df_web['article_summary'].dropna().str.len() if 'article_summary' in df_web.columns else pd.Series(dtype=float),
    'Web — article_summary_clean':df_web['article_summary_clean'].dropna().str.len() if 'article_summary_clean' in df_web.columns else pd.Series(dtype=float),
}

text_stats = pd.DataFrame({
    col: {'count': s.count(), 'mean': s.mean(), 'median': s.median(), 'min': s.min(), 'max': s.max(), 'std': s.std()}
    for col, s in text_cols.items() if len(s) > 0
}).T.round(1)

print('=== TEXT LENGTH STATISTICS (characters) ===')
print(text_stats.to_string())

In [ ]:
# ── Figure 3: Text length distribution boxplots ──────────────────────────────
valid_cols = {k: v for k, v in text_cols.items() if len(v) > 0}

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Figure 3 — Text Length Distribution per Field', fontsize=13, color=CHALK, fontweight='bold')

data_list = [v.values for v in valid_cols.values()]
labels    = list(valid_cols.keys())
bp = ax.boxplot(data_list, labels=labels, patch_artist=True, notch=False,
                medianprops=dict(color=HEAT, linewidth=2),
                whiskerprops=dict(color=SMOKE),
                capprops=dict(color=SMOKE),
                flierprops=dict(marker='o', color=NEON, alpha=0.5, markersize=3))

for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor('#dceeff' if i < 2 else '#dcf5e8')
    patch.set_edgecolor('#4488bb' if i < 2 else GREEN)

ax.set_ylabel('Character count')
ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=8)
ax.grid(axis='y')

plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/fig3_text_length.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 5. Gold Layer — Governance KPI Table

In [ ]:
# Load latest governance parquet
gov_files = sorted(glob.glob(f'{BASE}/datalake_gold/**/governance*.parquet', recursive=True))
df_gov = pd.read_parquet(gov_files[-1])

print(f'Governance file: {gov_files[-1].split("/")[-1]}')
print(f'Records: {len(df_gov)} | Columns: {list(df_gov.columns)}')
print(f"KPI categories: {df_gov['category'].unique().tolist()}")
print(f"KPI names:      {df_gov['kpi_name'].unique().tolist()}")

In [ ]:
# ── Full KPI table (screenshot-ready) ────────────────────────────────────────
kpi_display = df_gov[['kpi_name','category','source','field','value','unit','finding']].copy()
kpi_display['value'] = kpi_display['value'].round(2)
kpi_display = kpi_display.sort_values(['category','source','kpi_name']).reset_index(drop=True)

print('=== GOLD GOVERNANCE — FULL KPI TABLE ===')
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 100)
kpi_display

In [ ]:
# ── Figure 4: Governance KPI table rendered as matplotlib figure ─────────────
subset = df_gov[df_gov['kpi_name'] == 'null_rate'][['category','source','field','value','unit','finding']].copy()
subset['value'] = subset['value'].round(2)
subset = subset.sort_values(['source','value'], ascending=[True, False]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, max(4, len(subset) * 0.35 + 1.5)))
fig.suptitle('Figure 4 — Governance KPI Table: Null Rate per Field', fontsize=13, color=CHALK, fontweight='bold')
ax.axis('off')

col_labels = ['Category', 'Source', 'Field', 'Value', 'Unit', 'Finding']
col_widths  = [0.10, 0.08, 0.18, 0.07, 0.05, 0.52]

tbl = ax.table(
    cellText=subset.values,
    colLabels=col_labels,
    colWidths=col_widths,
    loc='center',
    cellLoc='left'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor('#cccccc')
    if row == 0:
        cell.set_facecolor('#2255aa')
        cell.set_text_props(color='white', fontweight='bold')
    else:
        val = subset.iloc[row-1]['value'] if col == 3 else None
        if col == 3 and val is not None:
            cell.set_facecolor('#ffd5d5' if float(val) > 20 else ('#fff4cc' if float(val) > 5 else '#f0fff4'))
        else:
            cell.set_facecolor('#f5f5f5' if row % 2 == 0 else '#ffffff')
        cell.set_text_props(color=CHALK)

plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/fig4_governance_kpi_table.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: fig4_governance_kpi_table.png')

In [ ]:
# ── Figure 5: KPI counts per category ────────────────────────────────────────
cat_counts = df_gov.groupby('category')['kpi_name'].nunique().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle('Figure 5 — Governance KPIs per Category', fontsize=13, color=CHALK, fontweight='bold')

palette = ['#4e9af1','#56c27a','#e8735a','#f0b429','#9b72cf','#5bc8c8']
bars = ax.barh(cat_counts.index, cat_counts.values, color=palette[:len(cat_counts)], edgecolor='none', height=0.5)

for bar, val in zip(bars, cat_counts.values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2, str(int(val)), va='center', fontsize=9, color=CHALK)

ax.set_xlabel('Number of unique KPI names')
ax.grid(axis='x')
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/fig5_kpi_categories.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ── Figure 6: Null rate bar chart from Gold governance ───────────────────────
null_gov = df_gov[df_gov['kpi_name'] == 'null_rate'].copy()
null_gov_nonzero = null_gov[null_gov['value'] > 0].sort_values('value', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Figure 6 — Null Rate per Field (from Gold Governance Parquet)', fontsize=13, color=CHALK, fontweight='bold')

if len(null_gov_nonzero) > 0:
    palette = [HEAT if v > 20 else NEON if v > 5 else '#4e9af1' for v in null_gov_nonzero['value']]
    bars = ax.bar(range(len(null_gov_nonzero)), null_gov_nonzero['value'],
                  color=palette, edgecolor='none', width=0.6)
    ax.set_xticks(range(len(null_gov_nonzero)))
    ax.set_xticklabels(
        [f"{row['source']}\n{row['field']}" for _, row in null_gov_nonzero.iterrows()],
        rotation=40, ha='right', fontsize=7
    )
    for bar, val in zip(bars, null_gov_nonzero['value']):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f'{val:.1f}%',
                ha='center', va='bottom', fontsize=7, color=CHALK)
    ax.axhline(y=5,  color=NEON, linestyle='--', alpha=0.7, linewidth=1, label='Warning (5%)')
    ax.axhline(y=20, color=HEAT, linestyle='--', alpha=0.7, linewidth=1, label='Critical (20%)')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'All fields: 0% null rate', ha='center', va='center',
            transform=ax.transAxes, color=GREEN, fontsize=14)

ax.set_ylabel('Null Rate (%)')
ax.grid(axis='y')
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/fig6_null_rate_gold.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 6. Gold Layer — Storytelling Summary

In [ ]:
# Load latest storytelling parquet
story_files = sorted(glob.glob(f'{BASE}/datalake_gold/**/storytelling*.parquet', recursive=True))
df_story = pd.read_parquet(story_files[-1])

print(f'Storytelling file: {story_files[-1].split("/")[-1]}')
print(f'Records: {len(df_story)} | Columns: {list(df_story.columns)}')
print(f"Aggregations: {df_story['aggregation'].unique().tolist()}")

In [ ]:
# ── Full storytelling table ───────────────────────────────────────────────────
print('=== GOLD STORYTELLING — FULL TABLE ===')
df_story[['aggregation','category','dimension_name','dimension_value','metric','value','label']]

In [ ]:
# ── Figure 7: Sentiment distribution donut ───────────────────────────────────
sent_dist = df_story[
    (df_story['aggregation'] == 'sentiment_distribution') &
    (df_story['metric'] == 'count')
].copy()

if len(sent_dist) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Figure 7 — Sentiment Distribution (Gold Storytelling)', fontsize=13, color=CHALK, fontweight='bold')

    ax = axes[0]
    label_color = {'positive': '#56c27a', 'negative': '#e8735a', 'neutral': '#aaaaaa'}
    labels_raw  = sent_dist['dimension_value'].str.lower().tolist()
    values_raw  = sent_dist['value'].tolist()
    colors_pie  = [label_color.get(l, '#4e9af1') for l in labels_raw]

    wedges, texts, autotexts = ax.pie(
        values_raw, labels=labels_raw, autopct='%1.1f%%',
        colors=colors_pie, startangle=90,
        pctdistance=0.8, wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
    )
    for t in texts:      t.set_color(CHALK)
    for at in autotexts: at.set_color('white'); at.set_fontweight('bold'); at.set_fontsize(9)
    ax.set_title('Sentiment Distribution', color=CHALK, fontsize=10, fontweight='bold')

    avg_row = df_story[
        (df_story['aggregation'] == 'sentiment_distribution') &
        (df_story['metric'] == 'avg_compound_score')
    ]
    if len(avg_row) > 0:
        avg_val = avg_row.iloc[0]['value']
        ax.text(0, 0, f'{avg_val:+.2f}\nVADER', ha='center', va='center',
                fontsize=10, color='#2255aa', fontweight='bold')

    ax2 = axes[1]
    bars = ax2.bar(labels_raw, values_raw, color=colors_pie, edgecolor='none', width=0.5)
    for bar, val in zip(bars, values_raw):
        ax2.text(bar.get_x() + bar.get_width()/2, val + 0.1, str(int(val)),
                 ha='center', va='bottom', fontsize=10, color=CHALK, fontweight='bold')
    ax2.set_ylabel('Number of articles')
    ax2.set_title('Article Count per Sentiment', color=CHALK, fontsize=10, fontweight='bold')
    ax2.grid(axis='y')

    plt.tight_layout()
    plt.savefig(f'{BASE}/notebooks/fig7_sentiment_dist.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()

In [ ]:
# ── Figure 8: Sentiment trend over time ──────────────────────────────────────
trend = df_story[
    (df_story['aggregation'] == 'sentiment_trend') &
    (df_story['metric'] == 'avg_sentiment')
].copy()

if len(trend) > 0:
    trend['week'] = pd.to_datetime(trend['dimension_value'], errors='coerce')
    trend = trend.dropna(subset=['week']).sort_values('week')

    fig, ax = plt.subplots(figsize=(12, 4))
    fig.suptitle('Figure 8 — Sentiment Trend Over Time (Weekly)', fontsize=13, color=CHALK, fontweight='bold')

    ax.plot(trend['week'], trend['value'], color='#2255aa', linewidth=2, marker='o',
            markersize=6, markerfacecolor=HEAT, markeredgecolor='none', zorder=3)
    ax.fill_between(trend['week'], trend['value'], alpha=0.10, color='#4e9af1')
    ax.axhline(y=0,     color=SMOKE, linewidth=0.8, linestyle='--', alpha=0.6)
    ax.axhline(y=0.05,  color=GREEN, linewidth=0.8, linestyle=':', alpha=0.7, label='Positive threshold (0.05)')
    ax.axhline(y=-0.05, color=HEAT,  linewidth=0.8, linestyle=':', alpha=0.7, label='Negative threshold (-0.05)')

    for _, row in trend.iterrows():
        ax.annotate(f"{row['value']:.2f}", xy=(row['week'], row['value']),
                    xytext=(0, 8), textcoords='offset points',
                    ha='center', fontsize=7, color=CHALK)

    ax.set_xlabel('Week')
    ax.set_ylabel('Avg VADER Compound Score')
    ax.legend(fontsize=8)
    ax.grid(True)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(f'{BASE}/notebooks/fig8_sentiment_trend.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()

In [ ]:
# ── Figure 9: Top keywords + sentiment ───────────────────────────────────────
kw = df_story[
    (df_story['aggregation'] == 'top_keywords') &
    (df_story['metric'] == 'count')
].copy().sort_values('value', ascending=False).head(20)

kw_sent = df_story[
    (df_story['aggregation'] == 'keyword_sentiment') &
    (df_story['metric'] == 'avg_compound')
].copy().set_index('dimension_value')['value']

if len(kw) > 0:
    kw['sentiment'] = kw['dimension_value'].map(kw_sent).fillna(0)
    kw_colors = ['#56c27a' if s > 0.05 else '#e8735a' if s < -0.05 else '#aaaaaa' for s in kw['sentiment']]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Figure 9 — Top Keywords: Frequency & Sentiment', fontsize=13, color=CHALK, fontweight='bold')

    ax = axes[0]
    ax.barh(kw['dimension_value'][::-1], kw['value'][::-1], color='#4e9af1', edgecolor='none', height=0.6)
    ax.set_xlabel('Frequency')
    ax.set_title('Keyword Frequency', color=CHALK, fontsize=10, fontweight='bold')
    ax.grid(axis='x')

    ax2 = axes[1]
    ax2.barh(kw['dimension_value'][::-1], kw['sentiment'][::-1], color=kw_colors[::-1], edgecolor='none', height=0.6)
    ax2.axvline(x=0,     color=CHALK, linewidth=0.8)
    ax2.axvline(x=0.05,  color=GREEN, linestyle=':', alpha=0.6, linewidth=0.8)
    ax2.axvline(x=-0.05, color=HEAT,  linestyle=':', alpha=0.6, linewidth=0.8)
    ax2.set_xlabel('Avg VADER Compound')
    ax2.set_title('Keyword Sentiment', color=CHALK, fontsize=10, fontweight='bold')
    ax2.grid(axis='x')

    plt.tight_layout()
    plt.savefig(f'{BASE}/notebooks/fig9_keywords.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()

In [ ]:
# ── Figure 10: Source comparison ─────────────────────────────────────────────
src = df_story[df_story['aggregation'] == 'source_comparison'].copy()

if len(src) > 0:
    pivot = src.pivot_table(index='dimension_value', columns='metric', values='value', aggfunc='first')
    print('Source comparison pivot:')
    print(pivot)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Figure 10 — Source Comparison: API vs Web Scraping', fontsize=13, color=CHALK, fontweight='bold')

    src_colors = ['#4e9af1', '#56c27a']
    for ax, col, label, color in zip(
        axes,
        ['record_count', 'avg_compound'] if 'record_count' in pivot.columns else [pivot.columns[0], pivot.columns[-1]],
        ['Record Count', 'Avg Sentiment'],
        src_colors
    ):
        if col in pivot.columns:
            bars = ax.bar(pivot.index, pivot[col], color=color, edgecolor='none', width=0.4)
            for bar, val in zip(bars, pivot[col]):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(pivot[col])*0.01,
                        f'{val:.2f}' if isinstance(val, float) else str(int(val)),
                        ha='center', va='bottom', fontsize=9, color=CHALK)
            ax.set_title(label, color=CHALK, fontsize=10, fontweight='bold')
            ax.grid(axis='y')

    plt.tight_layout()
    plt.savefig(f'{BASE}/notebooks/fig10_source_comparison.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()

---
## 7. Data Quality Findings Report

| # | Field / Column | Source | Issue Type | Severity | Frequency | Handling Strategy |
|---|---|---|---|---|---|---|
| 1 | `preparationMinutes`, `cookingMinutes` | API | High null rate (expected derived fields) | Medium | ~100% null | Coerced to `int32`, kept as-is — Spoonacular doesn't always expose these |
| 2 | `license`, `creditsText` | API | Partial nulls (~5–30%) | Low | Field-dependent | Retained — not used in NLP or aggregations |
| 3 | `author` | Web | Null for some articles | Low | ~10–20% | Retained — author not required for sentiment analysis |
| 4 | `article_summary` | Web | Variable length; some very short (<100 chars) | Medium | ~5% | Flagged in text_length KPI; kept — VADER handles short text |
| 5 | `extendedIngredients` | API | List serialized as string | Low | All records | Flattened to comma-separated `ingredient_names` in Silver |
| 6 | `categories` | Web | Some articles missing categories | Low | ~15% | Retained as empty string — category is secondary metadata |
| 7 | Full-row duplicates | API | Cross-file duplicates possible (daily runs) | Medium | Pipeline-dependent | Deduplication applied in Silver DAG |
| 8 | HTML tags in `summary` | API | Raw HTML fragments in summary text | High | ~80% of records | Stripped via BeautifulSoup in Silver preprocessing |

In [ ]:
# ── Descriptive stats — full Silver API ──────────────────────────────────────
numeric_api = df_api.select_dtypes(include=[np.number])
print('=== SILVER API — Numeric fields descriptive statistics ===')
numeric_api.describe().round(2)

In [ ]:
# ── Outlier detection — IQR method on numeric fields ─────────────────────────
def outlier_rate(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0: return 0.0
    outliers = ((series < q1 - 1.5*iqr) | (series > q3 + 1.5*iqr)).sum()
    return round(outliers / len(series) * 100, 2)

outlier_report = pd.DataFrame({
    col: {'outlier_rate_%': outlier_rate(numeric_api[col].dropna()),
          'mean': numeric_api[col].mean(),
          'std':  numeric_api[col].std(),
          'min':  numeric_api[col].min(),
          'max':  numeric_api[col].max()}
    for col in numeric_api.columns
}).T.round(2).sort_values('outlier_rate_%', ascending=False)

print('=== OUTLIER RATE per numeric field (IQR method) ===')
outlier_report

In [ ]:
# ── Figure 11: Outlier rate bar chart ────────────────────────────────────────
outlier_nonzero = outlier_report[outlier_report['outlier_rate_%'] > 0]['outlier_rate_%']

if len(outlier_nonzero) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.suptitle('Figure 11 — Outlier Rate per Numeric Field (IQR method, Silver API)', fontsize=12, color=CHALK, fontweight='bold')

    palette = [HEAT if v > 20 else NEON if v > 5 else '#4e9af1' for v in outlier_nonzero.values]
    bars = ax.bar(outlier_nonzero.index, outlier_nonzero.values, color=palette, edgecolor='none', width=0.6)
    for bar, val in zip(bars, outlier_nonzero.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f'{val:.1f}%',
                ha='center', va='bottom', fontsize=7, color=CHALK)

    ax.set_ylabel('Outlier Rate (%)')
    ax.set_xticklabels(outlier_nonzero.index, rotation=35, ha='right', fontsize=8)
    ax.axhline(y=10, color=SMOKE, linestyle='--', alpha=0.6, linewidth=1, label='10% threshold')
    ax.legend(fontsize=8)
    ax.grid(axis='y')
    plt.tight_layout()
    plt.savefig(f'{BASE}/notebooks/fig11_outlier_rates.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
else:
    print('No outliers detected in numeric fields.')

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('=== WORKSHOP 3 — SUMMARY ===')
print(f'Silver API   : {len(df_api):>5} records | {len(api_files)} files')
print(f'Silver Web   : {len(df_web):>5} records | {len(web_files)} files')
print(f'Gold Gov     : {len(df_gov):>5} KPI rows | {len(gov_files)} files')
print(f'Gold Story   : {len(df_story):>5} rows    | {len(story_files)} files')
print()
print(f'API fields w/ nulls   : {(nr_api > 0).sum()} / {len(nr_api)}')
print(f'Web fields w/ nulls   : {(nr_web > 0).sum()} / {len(nr_web)}')
print(f'API full duplicates   : {dup_api} ({dup_api/len(df_api)*100:.2f}%)')
print(f'Web full duplicates   : {dup_web} ({dup_web/len(df_web)*100:.2f}%)')
print(f'Governance KPI names  : {df_gov["kpi_name"].nunique()}')
print(f'Story aggregations    : {df_story["aggregation"].nunique()}')

if len(sent_dist) > 0:
    dom_sent = sent_dist.loc[sent_dist['value'].idxmax(), 'dimension_value']
    print(f'Dominant sentiment    : {dom_sent}')